# ARC-AGI Cookbook

A minimal, copy-paste-friendly tour of the ARC SDK: load a game, look at it, step it,
and let an LLM play.

**Workflow for editing a level**

1. Run the **Create editable copy** cell once. It copies the pristine game into
   `environment_files/ls20edit/` and loads it.
2. Edit `environment_files/ls20edit/<version>/ls20.py`.
3. Rerun the **Reload** cell to pick up your change.
4. Hit **Play** whenever you want to see the current version in action.

The copy cell is only needed on the first run — skip it afterwards so your edits are
never overwritten.


In [ ]:
%pip install -q "arc-agi==0.9.9" pillow matplotlib openai

## Setup

Paths, the editable game id, and the two helpers we reuse everywhere: `load()`
(re)loads a game from disk, and later `render`/`show`/`to_data_uri` turn an
observation into something we can look at or send to the model.


In [ ]:
from pathlib import Path
import json
import shutil

import arc_agi
import arc_agi.local_wrapper
from arc_agi import OperationMode

ENVIRONMENTS_DIR = "./environment_files"
EDIT_ID = "ls20edit"        # the editable copy we will work in


def load(game_id=EDIT_ID):
    """(Re)load a game from disk so edits to its .py are picked up."""
    arc_agi.local_wrapper._game_class_cache.clear()   # else the old class stays cached
    arcade = arc_agi.Arcade(
        operation_mode=OperationMode.OFFLINE,
        environments_dir=ENVIRONMENTS_DIR,
    )
    env = arcade.make(game_id)
    return env, env.reset()


## Create the editable copy (first run only)

Copies the downloaded game into `ls20edit/`, rewrites its `metadata.json` so the SDK
treats it as a separate game, and loads it. Safe to rerun: it only copies when the
folder does not exist yet, so your edits are never overwritten. Skip this cell after
the first run.


In [ ]:
# Where the original downloaded game lives, and where the copy will go.
SRC_FILE = next((Path(ENVIRONMENTS_DIR) / "ls20").glob("*/ls20.py"))
SRC_DIR = SRC_FILE.parent
VERSION = SRC_DIR.name
EDIT_DIR = Path(ENVIRONMENTS_DIR) / EDIT_ID / VERSION

if not EDIT_DIR.exists():
    shutil.copytree(SRC_DIR, EDIT_DIR)
    meta = json.loads((EDIT_DIR / "metadata.json").read_text())
    meta["game_id"] = f"{EDIT_ID}-{VERSION}"
    meta["local_dir"] = str(EDIT_DIR)
    (EDIT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))

env, obs = load()
print("edit this file:", EDIT_DIR / "ls20.py")
env.environment_info.game_id, obs.levels_completed


## Reload after editing

Changed `ls20edit/.../ls20.py`? Rerun this cell and the new version is live.


In [ ]:
env, obs = load()
obs.state, obs.levels_completed

## Look at the board

`obs` is the raw observation; `obs.frame[-1]` is the board as a numpy array of color
indices. `show(obs)` renders it as an image.


In [ ]:
import base64
import io

import matplotlib.pyplot as plt
from PIL import Image
from arc_agi.rendering import frame_to_rgb_array


def render(obs, scale=8):
    """Observation -> PIL image."""
    rgb = frame_to_rgb_array(steps=0, frame=obs.frame[-1], scale=scale)
    return Image.fromarray(rgb)


def show(obs, scale=8):
    """Observation -> inline matplotlib figure."""
    plt.figure(figsize=(3, 3))
    plt.imshow(render(obs, scale), interpolation="nearest")
    plt.axis("off")
    plt.show()


def to_data_uri(image):
    """PIL image -> base64 data URI for the chat API."""
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


show(obs)


## Step through the game

`step("ACTION1".."ACTION4")` applies one action, prints the state, and shows the new
board. Use it as your manual probe while editing a level.


In [ ]:
actions = {a.name: a for a in env.action_space}


def step(name):
    """Apply one action and redraw the board."""
    global obs
    obs = env.step(actions[name])
    print(obs.state, "| levels:", obs.levels_completed)
    show(obs)
    return obs


step("ACTION3")


In [ ]:
step("ACTION4")
step("ACTION1")
step("ACTION1")

## Let an LLM play

Same OpenAI-compatible endpoint as the API notebook, with the board sent as an image.
`play()` reloads the environment first, so it always runs your **current edited
version**.


In [ ]:
import re
from IPython.display import clear_output
from openai import OpenAI

MODEL = "lisa-pro"
client = OpenAI(
    base_url="https://chat-1.ki-awz.iisys.de/api",
    api_key=Path("api_key.txt").read_text().strip(),
)

PROMPT = """You play the ARC-AGI game ls20. The orange shape is your player; move it onto the gray cross.
Actions:
ACTION1 = up
ACTION2 = down
ACTION3 = left
ACTION4 = right
Reply with ONLY one action name, e.g. ACTION3."""

WORDS = {"UP": "ACTION1", "DOWN": "ACTION2", "LEFT": "ACTION3", "RIGHT": "ACTION4"}


def parse_action(text):
    """Pull an action out of the model's reply, tolerating 'left'/'up'/... too."""
    text = text.upper()
    match = re.search(r"ACTION[1-4]", text)
    if match:
        return match.group(0)
    for word, action in WORDS.items():
        if word in text:
            return action
    return None


def ask_llm_image(image, history):
    message = PROMPT + f"\nPrevious actions: {history}\nNext action:"
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": message},
            {"type": "image_url", "image_url": {"url": to_data_uri(image)}},
        ]}],
    )
    return parse_action(response.choices[0].message.content)


def ask_llm_text(message):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": message},
        ]}],
    )
    return response.choices[0].message.content

print(ask_llm_text("Are you ready?"))

In [ ]:
def play(max_steps=20):
    """Reload the current game and let the LLM play until it wins or runs out of steps."""
    global obs
    env, obs = load()
    acts = {a.name: a for a in env.action_space}
    history = []

    for i in range(max_steps):
        clear_output(wait=True)
        print(f"Step {i + 1}/{max_steps} | {obs.state} | levels: {obs.levels_completed}")
        print("History:", history)
        show(obs)

        action = ask_llm_image(render(obs), history)
        if action is None:
            print("No valid action returned. Stopping.")
            break

        history.append(action)
        print("chose:", action)
        obs = env.step(acts[action])

        if obs.state.name in {"WIN", "GAME_OVER"}:
            break

    return history


play(20)